### Imports

In [41]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from skimage.measure import shannon_entropy
from skimage.feature import local_binary_pattern, hog
from tqdm import tqdm
from natsort import natsorted

SEED = 42

### Paths

In [40]:
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/ml_features/"
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)

CLASSES = [
    "Normal",
    "COVID",
    "Lung_Opacity",
    "Viral Pneumonia"
]


### Statistical features

In [4]:
def extract_stat_features(image):
    """
    Extraction des features sur les pixels non nuls uniquement.
    """

    # pixels pulmonaires uniquement
    pixels = image[image > 0]

    if len(pixels) == 0:
        return None

    # calcul des gradients
    grad_x = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    grad_pixels = grad_mag[image > 0]

    features = {
        "mean": pixels.mean(),
        "std": pixels.std(),
        "min": pixels.min(),
        "max": pixels.max(),
        "median": np.median(pixels),
        "p5": np.percentile(pixels, 5),
        "p25": np.percentile(pixels, 25),
        "p75": np.percentile(pixels, 75),
        "p95": np.percentile(pixels, 95),
        "skewness": skew(pixels), # asymétrie de la distribution des pixels
        "kurtosis": kurtosis(pixels), # degré de concentration/présence de valeurs extrêmes dans la distribution
        "entropy": shannon_entropy(pixels), # entropie calculée sur les pixels pulmonaires uniquement
        "laplacian_variance": cv2.Laplacian(image.astype(np.float32), cv2.CV_32F)[image > 0].var(), # détection de variations brusques d'intensité (contours), sharpness, netteté
        "gradient_magnitude_mean": grad_pixels.mean(), # gradient moyenne
        "gradient_magnitude_std": grad_pixels.std(),# gradients écart-type
    }

    return features

In [ ]:
# Extraction des features statistiques et enregistrement dans un csv

for contrast_adjustment in ["clahe", "no_clahe"]:
    print(f"\n#####   {contrast_adjustment}   #####\n")
    data = []
    for class_name in CLASSES:
        image_dir = os.path.join(PROCESSED_DATA_DIR, contrast_adjustment, class_name)
        for image_name in tqdm(natsorted(os.listdir(image_dir)), desc=class_name):
            image_path = os.path.join(image_dir, image_name)
            image = cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)

            if image is None:
                continue

            features = extract_stat_features(image)

            if features is None:
                continue

            features["class"] = class_name
            features["image_name"] = image_name

            data.append(features)
             
        print(len(data[0])-2, "features extraites")

    df_features = pd.DataFrame(data)
    cols = ["image_name"] + [col for col in df_features.columns if col != "image_name"]
    df_features = df_features[cols]
    df_features.to_csv(f"{FEATURES_DIR}/features_stat_{contrast_adjustment}.csv", index=False)

# display(df_features.head())


#####   clahe   #####



Normal: 100%|██████████| 10192/10192 [00:49<00:00, 206.64it/s]


15 features extraites


COVID: 100%|██████████| 3616/3616 [00:17<00:00, 204.51it/s]


15 features extraites


...

15 features extraites


Lung_Opacity: 100%|██████████| 6012/6012 [00:28<00:00, 207.61it/s]


15 features extraites


Viral Pneumonia: 100%|██████████| 1345/1345 [00:05<00:00, 236.88it/s]


15 features extraites


### LBP features

In [37]:
# paramètres LBP
RADIUS = 3
N_POINTS = 8 * RADIUS
METHOD = "uniform"

def extract_lbp_features(image):
    """
    image : grayscale segmentée, uint8
    """
    lbp = local_binary_pattern(
        image,
        P=N_POINTS,
        R=RADIUS,
        method=METHOD
    )

    mask = image > 0
    lbp_pixels = lbp[mask]

    if len(lbp_pixels) == 0:
        return None

    n_bins = N_POINTS + 2  # pour method='uniform'

    hist, _ = np.histogram(
        lbp_pixels,
        bins=n_bins,
        range=(0, n_bins),
        density=True
    )
    
    features = {
        f"lbp_{i}": hist[i]
        for i in range(len(hist))
    }

    return features

In [ ]:
# Extraction des features LBP et enregistrement dans un csv

for contrast_adjustment in ["clahe", "no_clahe"]:
    print(f"\n#####   {contrast_adjustment}   #####\n")
    data = []
    for class_name in CLASSES:
        image_dir = os.path.join(PROCESSED_DATA_DIR, contrast_adjustment, class_name)
        for image_name in tqdm(natsorted(os.listdir(image_dir)), desc=class_name):
            image_path = os.path.join(image_dir, image_name)
            image = cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)

            if image is None:
                continue

            features = extract_lbp_features(image)

            if features is None:
                continue

            features["class"] = class_name
            features["image_name"] = image_name

            data.append(features)
            
        print(len(data[0])-2, "features extraites")

    df_features = pd.DataFrame(data)
    cols = ["image_name"] + [col for col in df_features.columns if col != "image_name"]
    df_features = df_features[cols]
    df_features.to_csv(f"{FEATURES_DIR}/features_lbp_{contrast_adjustment}.csv", index=False)

# display(df_features.head())


#####   clahe   #####



Normal: 100%|██████████| 10192/10192 [08:00<00:00, 21.20it/s]


26 features extraites


COVID: 100%|██████████| 3616/3616 [05:41<00:00, 10.59it/s]


26 features extraites


...

26 features extraites


Lung_Opacity: 100%|██████████| 6012/6012 [14:59<00:00,  6.69it/s]


26 features extraites


Viral Pneumonia: 100%|██████████| 1345/1345 [03:12<00:00,  6.98it/s]


26 features extraites


### HOG features

In [46]:
# paramètres HOG
HOG_IMAGE_SIZE = 160
ORIENTATION = 9
PIXELS_PER_CELL = (32, 32)
CELLS_PER_BLOCK = (2, 2)
BLOCK_NORM = "L2-Hys"

def extract_hog_features(image):
    """
    image : grayscale segmentée, paddée, resizée, uint8
    """

    image = cv2.resize(image, (HOG_IMAGE_SIZE, HOG_IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    features = hog(
        image,
        orientations=ORIENTATION,
        pixels_per_cell=PIXELS_PER_CELL,
        cells_per_block=CELLS_PER_BLOCK,
        block_norm=BLOCK_NORM,
        visualize=False,
        feature_vector=True
    )

    features = {
        f"hog_{i}": value
        for i, value in enumerate(features)
    }
    
    return features

In [48]:
# Extraction des features HOG et enregistrement dans un csv

for contrast_adjustment in ["clahe", "no_clahe"]:
    print(f"\n#####   {contrast_adjustment}   #####\n")
    data = []
    for class_name in CLASSES:
        image_dir = os.path.join(PROCESSED_DATA_DIR, contrast_adjustment, class_name)
        for image_name in tqdm(natsorted(os.listdir(image_dir)), desc=class_name):
            image_path = os.path.join(image_dir, image_name)
            image = cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)

            if image is None:
                continue

            features = extract_hog_features(image)

            if features is None:
                continue

            features["class"] = class_name
            features["image_name"] = image_name

            data.append(features)
            
        print(len(data[0])-2, "features extraites")

    df_features = pd.DataFrame(data)
    cols = ["image_name"] + [col for col in df_features.columns if col != "image_name"]
    df_features = df_features[cols]
    df_features.to_csv(f"{FEATURES_DIR}/features_hog_{contrast_adjustment}.csv", index=False)

# display(df_features.head())


#####   clahe   #####



Normal: 100%|██████████| 10192/10192 [00:26<00:00, 378.85it/s]


576 features extraites


COVID: 100%|██████████| 3616/3616 [00:09<00:00, 379.58it/s]


576 features extraites


...

576 features extraites


Lung_Opacity: 100%|██████████| 6012/6012 [00:14<00:00, 404.65it/s]


576 features extraites


Viral Pneumonia: 100%|██████████| 1345/1345 [00:03<00:00, 406.67it/s]


576 features extraites
